# TTM-Small-4M × EmbeddingGemma 2 — thử nghiệm GPU 1.024 bước A/B/C/D

**Mục tiêu:** xác định EmbeddingGemma 2 có giúp TTM-Small tốt hơn sau một lượt **thích nghi bổ sung** có đối chứng. Mỗi cấu hình nhận **1.024 bước × 4.096 nhãn TRAIN/bước**, cùng lịch lấy mẫu. Tất cả bắt đầu từ **TTM bước 1.024 đã huấn luyện**, không phải from-scratch. Không đọc TEST, không RAG, không huấn luyện lại mô hình nguồn.

**Bước thực hiện:** trong Colab Free chọn **Runtime → Change runtime type → T4 GPU**, khi TTM-Small T5 không dùng GPU. Chạy toàn notebook **một lần**. Các checkpoint của A/B/C/D được ghi tại 256/512/768/1.024 và VALIDATION chỉ chấm sau khi toàn bộ bốn arm đạt mốc cuối.

Nếu Colab gián đoạn sau khi đã có `CLAIM.json`, đổi **duy nhất** `RESUME=True` ở ô mã và chạy lại. Dùng lại chính RUN_ID, không đổi mã nguồn, dữ liệu hay optimizer. Nếu đã có `RESULT.json`, không chạy lại. **Bạn không cần tải ZIP thủ công**, notebook lấy mã đã khóa trên GitHub và dữ liệu riêng tư đã lưu ở Google Drive.

[Giao thức khoa học đã khóa](https://github.com/minhtri22/TokenToModel-Lab/blob/research/ttm-small4m-embeddinggemma2-token-swap-minimal-proof-20261109/docs/pilot/TTM_SMALL4M_GEMMA2_DIRECT_SWAP_FINAL_1024_GPU_EQUAL_EXPOSURE_PREREG_20261109.md).


In [ ]:
# TTM-Small × EmbeddingGemma2: one 1024x4096 adaptation, four arms, same RUN_ID.
RESUME = False  # Only after Colab interruption with existing CLAIM: change to True.

import hashlib, subprocess, sys, urllib.request, zipfile
from pathlib import Path
import torch
from google.colab import drive

RUN_ID = "ttm_small4m_gemma2_directswap_1024x4096_t4_gpu_20261109_v1"
COMMIT = "a22a7d3913b049b93006efa9475ca574ab2877a9"
ZIP_SHA = "e544c645635a28fe140802421d03a7928a7f0446d2df33bbfe4ec129e3a6ea27"
RUNNER_SHA = "00416c0ff5b2afdcca2780a20315dd2da0af93554cd2b32b6c94efb630dc5d3b"
SOURCE_HASHES = {
  "p02_exact.py": "e0fcac285918b66b13c8f694e6d625a0691c9b59221b43043e63f46b25ddcc07",
  "ttm_embeddinggemma2_swap.py": "d11da61e9d7bb040f3263e680f026817a513553c186ab243e184ffaf6e68cd39",
  "ttm_directswap1024_gpu_once.py": RUNNER_SHA
}
def sha(data): return hashlib.sha256(data).hexdigest()
if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("Requires a Colab T4 GPU; no CLAIM created")
print("TTM_GPU1024_GPU="+torch.cuda.get_device_name(0))
drive.mount("/content/drive")
root=Path("/content/drive/MyDrive")
pack=root/"TTM"/"ttm-small-pilot"/"t4_private_pack_20261108_v1"
base=root/"TTM"/"ttm-small-pilot"/"t4_science_4m_1024_a2_20261108_v1"/"checkpoints"/"step_1024"/"checkpoint.pt"
candidate=root/"TTM_Small4M_EmbeddingSwap"/"real_token_mapping_step1024_once_20261109_v1"/"candidate_initialization.pt"
out=root/"TTM_Small4M_EmbeddingSwap"/RUN_ID
src=Path("/content/ttm_gemma2_directswap1024_source")
scratch=Path("/content/ttm_gemma2_directswap1024_scratch")
for label,file in [("T4 checkpoint",base),("Gemma candidate",candidate),
                   ("TRAIN/VAL pack",pack/"T4_PRIVATE_TRAIN_VAL_WINDOWS.npz"),
                   ("pack manifest",pack/"T4_PRIVATE_PACK_MANIFEST.json")]:
    if not file.is_file(): raise FileNotFoundError(label+": "+str(file))
if out.exists() and not RESUME: raise RuntimeError("CLAIM may exist; use RESUME=True for same RUN_ID")
if not out.exists() and RESUME: raise RuntimeError("No CLAIM to resume; use RESUME=False")
src.mkdir(parents=True,exist_ok=True)
zip_path=src/"frozen_source.zip"
url=("https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/"+COMMIT+
     "/tools/embeddinggemma2_swap/TTM_GEMMA2_DIRECTSWAP_GPU1024_SOURCE_20261109.zip")
if zip_path.is_file(): raw=zip_path.read_bytes()
else:
    with urllib.request.urlopen(url,timeout=120) as fd: raw=fd.read()
if sha(raw)!=ZIP_SHA: raise RuntimeError("SOURCE_ZIP_SHA256_MISMATCH_NO_CLAIM")
if not zip_path.exists(): zip_path.write_bytes(raw)
with zipfile.ZipFile(zip_path) as z:
    if set(z.namelist())!=set(SOURCE_HASHES): raise RuntimeError("FROZEN_FILE_LIST_MISMATCH")
    for name,digest in SOURCE_HASHES.items():
        data=z.read(name)
        if sha(data)!=digest: raise RuntimeError("SOURCE_HASH_MISMATCH:"+name)
        dest=src/name
        if dest.exists() and sha(dest.read_bytes())!=digest: raise RuntimeError("EXISTING_SOURCE_MISMATCH:"+name)
        if not dest.exists(): dest.write_bytes(data)
cmd=[sys.executable,str(src/"ttm_directswap1024_gpu_once.py"),
     "--base",str(base),"--candidate",str(candidate),
     "--pack",str(pack/"T4_PRIVATE_TRAIN_VAL_WINDOWS.npz"),
     "--manifest",str(pack/"T4_PRIVATE_PACK_MANIFEST.json"),
     "--source-dir",str(src),"--output",str(out),"--scratch",str(scratch),
     "--runner-sha256",RUNNER_SHA]
if RESUME:cmd.append("--resume")
print("TTM_GPU1024_RUN_ID="+RUN_ID)
print("TTM_GPU1024_SCOPE=4_ARMS_1024_STEPS_4096_TARGETS_EACH_TRAIN_ONLY")
ret=subprocess.run(cmd,check=False)
if ret.returncode:raise RuntimeError("INCOMPLETE_OR_ERROR_EXIT_"+str(ret.returncode))
print("TTM_GPU1024_DRIVE_RESULT="+str(out/"RESULT.json"))
